In [ ]:
import sys
sys.path.append('/workspace/python')

import duckdb
import polars as pl
import json
import utils_functions as utils

In [34]:
#Connect to DuckDB
con = duckdb.connect('/workspace/data/pipeline.duckdb')
print('Connected to DuckDB!')

Connected to DuckDB!


In [ ]:
last_batch = con.execute('SELECT MAX(batch_id) FROM raw.api_data').fetchone()[0]

**MODIFY THE LINE OF CODE BELOW FOR DEBUGGING PURPOSES, OTHERWISE LEAVE IT ALONE SO IT ALWAYS LOADS THE LAST BATCH**

In [ ]:
historical_batch_to_load = last_batch

In [ ]:
# Adding the bronze table into a polars dataframe for further processing
df = con.execute(f"""
                 SELECT id, batch_id, page, raw_json 
                 FROM raw.api_data
                 WHERE batch_id = {historical_batch_to_load}
                 """).pl()

In [ ]:
#Taking only the data column
df = df.with_columns([
    pl.col('raw_json')
      .map_elements(
          lambda x: [json.dumps(anime) for anime in json.loads(x)['data']], 
          return_dtype=pl.List(pl.String)
      )
      .alias('anime_list')
])

In [ ]:
#Splitting each json with 25 animes into 25 rows with 1 anime each
df = df.explode('anime_list')

In [ ]:
df = df.drop('raw_json').rename({
    'anime_list': 'anime_json',
    'id' : 'raw_id'
    })

In [ ]:
records = [json.loads(x) for x in df['anime_json'].to_list()]
df_flattened = pl.json_normalize(records)

In [ ]:
# Removing unnecessary columns
df_flattened = df_flattened[[
    'mal_id', 'title', 'title_english', 'type', 'source', 'episodes', 'status', 'duration', 'score', 'scored_by', 'rank', 'popularity',
    'members', 'favorites', 'year', 'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
    ]]

In [ ]:
# Renaming columns
df_flattened = df_flattened.rename({
    'mal_id': 'anime_id',
    'episodes': 'no_of_episodes'
})

In [ ]:
df_flattened = df_flattened.with_columns([
    pl.col('producers')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('producer_id'),
              pl.element().struct.field('name').alias('producer_name')
          )
      )
      .alias('producers'),
    
    pl.col('licensors')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('licensor_id'),
              pl.element().struct.field('name').alias('licensor_name')
          )
      )
      .alias('licensors'),

    pl.col('studios')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('studio_id'),
              pl.element().struct.field('name').alias('studio_name')
          )
      )
      .alias('studios'),

    pl.col('genres')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('genre_id'),
              pl.element().struct.field('name').alias('genre_name')
          )
      )
      .alias('genres'),

    pl.col('themes')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('theme_id'),
              pl.element().struct.field('name').alias('theme_name')
          )
      )
      .alias('themes'),

    pl.col('demographics')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('demographic_id'),
              pl.element().struct.field('name').alias('demographic_name')
          )
      )
      .alias('demographics')
])


In [ ]:
concatted_df = pl.concat([df.drop('anime_json'), df_flattened], how='horizontal')

In [ ]:
# Extracting bridge tables for genres, themes, demographics, studios, producers and licensors
links_anime_genres       = utils.extract_bridge_table(concatted_df, 'genres',       'genre_id',       'genre_name')
links_anime_themes       = utils.extract_bridge_table(concatted_df, 'themes',       'theme_id',       'theme_name')
links_anime_demographics = utils.extract_bridge_table(concatted_df, 'demographics', 'demographic_id', 'demographic_name')
links_anime_studios      = utils.extract_bridge_table(concatted_df, 'studios',      'studio_id',      'studio_name')
links_anime_producers    = utils.extract_bridge_table(concatted_df, 'producers',    'producer_id',    'producer_name')
links_anime_licensors    = utils.extract_bridge_table(concatted_df, 'licensors',    'licensor_id',    'licensor_name')

In [ ]:
# Dropping the list[struct] columns as we already have all that information in the links tables
silver_anime = concatted_df.drop([
    'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
])

In [ ]:
#Dropping duplicates:

links_anime_genres = links_anime_genres.unique(subset=['anime_id', 'genre_id'], keep='last')
links_anime_themes = links_anime_themes.unique(subset=['anime_id', 'theme_id'], keep='last')
links_anime_demographics = links_anime_demographics.unique(subset=['anime_id', 'demographic_id'], keep='last')
links_anime_studios = links_anime_studios.unique(subset=['anime_id', 'studio_id'], keep='last')
links_anime_producers = links_anime_producers.unique(subset=['anime_id', 'producer_id'], keep='last')
links_anime_licensors = links_anime_licensors.unique(subset=['anime_id', 'licensor_id'], keep='last')

silver_anime = silver_anime.unique(subset='anime_id', keep='last')

**The cells below overwrite the merge tables in DuckDB with our dataframes.** 

**You can run them one by one for debugging purposes, but the notebook will run all of them when called.**

In [ ]:
#import importlib
#importlib.reload(utils)

utils.overwrite_table(con, silver_anime, 'curated.anime_merge')

In [ ]:
utils.overwrite_table(con, links_anime_genres, 'curated.links_anime_genres_merge')

In [ ]:
utils.overwrite_table(con, links_anime_themes, 'curated.links_anime_themes_merge')

In [ ]:
utils.overwrite_table(con, links_anime_demographics, 'curated.links_anime_demographics_merge')

In [ ]:
utils.overwrite_table(con, links_anime_studios, 'curated.links_anime_studios_merge')

In [ ]:
utils.overwrite_table(con, links_anime_producers, 'curated.links_anime_producers_merge')

In [ ]:
utils.overwrite_table(con, links_anime_licensors, 'curated.links_anime_licensors_merge')

**The cells below upsert the merge tables into the destination curated/silver tables in DuckDB.** 

**You can run them one by one for debugging purposes, but the notebook will run all of them when called.**

In [ ]:
utils.upsert_table(con, 'curated.anime_merge',              'curated.anime',              ['anime_id'])

In [ ]:
utils.upsert_table(con, 'curated.links_anime_genres_merge',       'curated.links_anime_genres',       ['anime_id', 'genre_id'])

In [ ]:
utils.upsert_table(con, 'curated.links_anime_themes_merge',       'curated.links_anime_themes',       ['anime_id', 'theme_id'])

In [ ]:
utils.upsert_table(con, 'curated.links_anime_demographics_merge', 'curated.links_anime_demographics', ['anime_id', 'demographic_id'])

In [ ]:
utils.upsert_table(con, 'curated.links_anime_studios_merge',      'curated.links_anime_studios',      ['anime_id', 'studio_id'])

In [ ]:
utils.upsert_table(con, 'curated.links_anime_producers_merge',    'curated.links_anime_producers',    ['anime_id', 'producer_id'])

In [ ]:
utils.upsert_table(con, 'curated.links_anime_licensors_merge',    'curated.links_anime_licensors',    ['anime_id', 'licensor_id'])

In [37]:
con.close()